In [ ]:
# 7_label_clusters.ipynb
#
# For each cluster produced by 6_cluster.ipynb, calls the OpenAI Chat API
# to generate a short title and 2-3 sentence description, then saves the
# enriched results to data/6_cluster/<name>_described.csv.
#
# Requires OPENAI_API_KEY in environment or .env file.

import sys, os, time, json
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from tqdm import tqdm

import importlib
from data_processing.config_paths     import DATA_FOLDER
import data_processing.config_variables as _cv
import data_processing.config_cluster   as _cc
importlib.reload(_cv)
importlib.reload(_cc)

from data_processing.config_variables import VARIABLE_MAP
from data_processing.config_cluster   import WAVE
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
MODEL       = "gpt-4o-mini"
CLUSTER_CSV = Path(f"../{DATA_FOLDER}/6_cluster/LA_london_clusters.csv")
OUTPUT_CSV  = CLUSTER_CSV.parent / (CLUSTER_CSV.stem + "_described.csv")
MAX_RETRIES = 3
RETRY_DELAY = 5

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model:       {MODEL}")
print(f"Cluster CSV: {CLUSTER_CSV}")
print(f"Output CSV:  {OUTPUT_CSV}")

# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CLUSTER_CSV)
META_COLS = {'tribe_label', 'size', 'unit_id', 'cluster_level', 'group'}
print(f"Loaded {len(df)} cluster rows across {df['unit_id'].nunique()} units")

# ── Prompts ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a social researcher specialising in UK population demographics.
You will be given the statistical profile of a population cluster derived from the UK Household
Longitudinal Study (UKHLS), projected onto a local authority area.

For each cluster, respond with a JSON object containing exactly two keys:
  \"title\"       — a vivid, memorable 3-5 word name for this persona group (e.g. \"Settled suburban professionals\")
  \"description\" — 2-3 sentences describing the defining characteristics of this group
                  in plain English, highlighting what makes them distinct.

Notes on the data:
- \"Highest qualification\" is an average code: 1=Degree, 2=Other Higher, 3=A-Level, 4=GCSE, 5=Other/None. Lower = higher qualification.
- \"Social class (NS-SEC 8)\" is an average code: 1=Higher managerial, 8=Routine. Lower = higher class.
- \"Mental/Physical health score (SF-12)\" scale 0-100; higher = better health.
- Neighbourhood Cohesion Index 1-5; higher = stronger community.
- Local services ratings 1-5; higher = better satisfaction.
Respond with valid JSON only — no markdown, no explanation outside the JSON."""


def build_prompt(row: pd.Series) -> str:
    lines = [
        f"Unit (Local Authority): {row.get('unit_id', 'unknown')}",
        f"Employment group:       {row.get('group', 'Unknown')}",
        f"Cluster label:          {row['tribe_label']}",
        f"Population size:        {int(row['size']):,}",
        "",
        "--- Cluster statistics ---",
    ]
    for col in [c for c in row.index if c not in META_COLS]:
        val = row[col]
        if pd.isna(val):
            continue
        if isinstance(val, float) and val == int(val):
            val_str = str(int(val))
        elif isinstance(val, float):
            val_str = f"{val:.1f}"
        else:
            val_str = str(val)
        lines.append(f"  {col}: {val_str}")
    return "\n".join(lines)


# ── Call ChatGPT for each cluster ─────────────────────────────────────────────
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Labelling clusters"):
    response_text = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_prompt(row)},
                ],
                response_format={"type": "json_object"},
                temperature=0.7,
                max_tokens=300,
            )
            response_text = resp.choices[0].message.content
            break
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"\n  Error on {row['tribe_label']} ({row['unit_id']}): {e}  — retrying in {wait}s")
            time.sleep(wait)

    title, desc = "", ""
    if response_text:
        try:
            parsed = json.loads(response_text)
            title = parsed.get("title", "").strip()
            desc  = parsed.get("description", "").strip()
        except json.JSONDecodeError:
            desc = response_text.strip()

    results.append({
        "unit_id":         row["unit_id"],
        "tribe_label":     row["tribe_label"],
        "group":           row.get("group", ""),
        "gpt_title":       title or None,
        "gpt_description": desc  or None,
    })

print(f"\nCompleted {len(results)} clusters")
print(f"  Successful: {sum(1 for r in results if r['gpt_title'])}")
print(f"  Failed:     {sum(1 for r in results if not r['gpt_title'])}")

# ── Merge back and save ───────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
enriched = df.merge(
    results_df[["unit_id", "tribe_label", "gpt_title", "gpt_description"]],
    on=["unit_id", "tribe_label"],
    how="left",
)
enriched.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(enriched)} rows to {OUTPUT_CSV}")
enriched[["unit_id", "group", "tribe_label", "size", "gpt_title", "gpt_description"]].head(12)

# -- Copy data files into api/ for deployment --------------------------------
import shutil

API_DATA_DIR     = Path('..') / 'api' / 'data'
API_CLUSTERS_DIR = API_DATA_DIR / '6_cluster'
API_GEO_DIR      = API_DATA_DIR / '0_raw'
API_CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
API_GEO_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(CLUSTER_CSV, API_CLUSTERS_DIR / CLUSTER_CSV.name)
print(f'Copied {CLUSTER_CSV.name}  ->  api/data/6_cluster/')

shutil.copy(OUTPUT_CSV, API_CLUSTERS_DIR / OUTPUT_CSV.name)
print(f'Copied {OUTPUT_CSV.name}  ->  api/data/6_cluster/')

GEO_SRC = Path(f'../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv')
if GEO_SRC.exists():
    shutil.copy(GEO_SRC, API_GEO_DIR / 'admin_geography_mappings.csv')
    print('Copied admin_geography_mappings.csv  ->  api/data/0_raw/')

# -- Copy data files into api/ for deployment --------------------------------
import shutil

API_DATA_DIR     = Path('..') / 'api' / 'data'
API_CLUSTERS_DIR = API_DATA_DIR / '6_cluster'
API_GEO_DIR      = API_DATA_DIR / '0_raw'
API_CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
API_GEO_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(CLUSTER_CSV, API_CLUSTERS_DIR / CLUSTER_CSV.name)
print(f'Copied {CLUSTER_CSV.name}  ->  api/data/6_cluster/')

shutil.copy(OUTPUT_CSV, API_CLUSTERS_DIR / OUTPUT_CSV.name)
print(f'Copied {OUTPUT_CSV.name}  ->  api/data/6_cluster/')

GEO_SRC = Path(f'../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv')
if GEO_SRC.exists():
    shutil.copy(GEO_SRC, API_GEO_DIR / 'admin_geography_mappings.csv')
    print('Copied admin_geography_mappings.csv  ->  api/data/0_raw/')
